# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List available Record Sets and their IDs
print("Available Record Sets and their @id fields:\n")
for rs in metadata.record_sets:
    print(f"- Name: {rs.name}, @id: {rs.id}")
    print(f"  Description: {rs.description}")
    # List fields/columns for each record set
    print("  Fields (with their @id):")
    for field in rs.fields:
        print(f"    - {field.name}: {field.id}")
    print()

## 3. Data Extraction
Load data from one or more specific record sets into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Identify all record set @ids
record_set_ids = [rs.id for rs in metadata.record_sets]

dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)

if record_set_ids:
    print(f'First RecordSet: {record_set_ids[0]}')
    print('Columns:')
    print(dataframes[record_set_ids[0]].columns.tolist())
    display(dataframes[record_set_ids[0]].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes filtering, normalization, and grouping with the fields identified by their `@id`.

In [ ]:
# Choose record set and numeric field by @id (use previous overview)
# For demonstration, select the first available record set and first numeric field
if record_set_ids:
    selected_record_set_id = record_set_ids[0]
    df = dataframes[selected_record_set_id]
    # Try to infer numeric fields by dtype
    numeric_field_candidates = df.select_dtypes(include=['number']).columns.tolist()
    if numeric_field_candidates:
        numeric_field = numeric_field_candidates[0]
        threshold = df[numeric_field].mean()  # Set threshold as mean for demonstration
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold}:")
        display(filtered_df.head())

        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try grouping by a non-numeric field, if exists
        group_field_candidates = df.select_dtypes(exclude=['number']).columns.tolist()
        group_field = group_field_candidates[0] if group_field_candidates else None
        if group_field is not None:
            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
            print(f"Grouped data by {group_field}:")
            display(grouped_df.head())
        else:
            print('No non-numeric field available to group by.')
    else:
        print('No numeric fields found in the DataFrame.')
else:
    print('No record sets available in the dataset.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt

# Example: Plot histogram of the numeric field and a boxplot by a group field (if available)
if record_set_ids and numeric_field_candidates:
    plt.figure(figsize=(8,4))
    df[numeric_field].hist(bins=30)
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    if group_field is not None:
        plt.figure(figsize=(10,5))
        df.boxplot(column=numeric_field, by=group_field, rot=90)
        plt.title(f'{numeric_field} by {group_field}')
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.


- The dataset was loaded using the `mlcroissant` library via its Croissant JSON-LD schema.
- Record sets and fields were explored using their `@id` references for reproducibility.
- Simple EDA including filtering, normalization, grouping, and visualization was demonstrated.
- For advanced analyses and interpretation, refer to the dataset's FAIR documentation and the `mlcroissant` documentation for further APIs and usage patterns.